# Dashboard Interativo – Eficiência Energética e Mobilidade Elétrica

Este dashboard foi desenvolvido com o objetivo de proporcionar uma visualização interativa dos principais indicadores relacionados com a eficiência da iluminação pública e a capacidade da rede elétrica para suportar a integração de carregadores de veículos elétricos (VE).

Através deste painel, é possível analisar, ao nível do concelho, o impacto da modernização da iluminação pública para tecnologia LED na libertação de potência e na viabilidade de instalação de infraestrutura de carregamento.

O dashboard integra as seguintes componentes principais:

Perfis horários de consumo da iluminação pública, comparando o cenário atual com o cenário após modernização para tecnologia LED;
Avaliação da capacidade instalada e da capacidade disponível nos Postos de Transformação de Distribuição (PTD);
Estimativa da potência libertada resultante da substituição de tecnologias ineficientes;
Simulação de cenários de integração de carregadores de veículos elétricos e respetivo impacto na carga da rede;
Representação geográfica dos PTDs, permitindo identificar zonas com potencial para instalação de pontos de carregamento.

Este painel constitui uma ferramenta de apoio à decisão, permitindo explorar de forma intuitiva os resultados obtidos nas análises anteriores e avaliar diferentes cenários energéticos ao nível local.

In [13]:
import os
from pathlib import Path

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, Markdown

# Garantir que o notebook encontra a pasta do projeto
notebook_dir = Path.cwd()
if notebook_dir.name == "src":
    os.chdir(notebook_dir.parent)

# Caminhos
caminho_dataset = Path("data/dataset_final.csv")
caminho_ptd = Path("data/PTD_data.xlsx")

# Dataset final
df_dashboard = pd.read_csv(caminho_dataset)

df_dashboard["Viabilidade_VE"] = np.where(
    df_dashboard["D"] >= 0,
    "Viável",
    "Não Viável"
)

# PTDs
ptd_data = pd.read_excel(caminho_ptd)
ptd_data.columns = ptd_data.columns.str.strip()

print("Dataset carregado com sucesso.")
print(f"Número de concelhos: {len(df_dashboard)}")
display(df_dashboard.head())

Dataset carregado com sucesso.
Número de concelhos: 278


,Distrito,Concelho,CodDistritoConcelho,P_IP_TOTAL,P_IP_Inef,Rate_Ineficiencia,Cap_PTD,Util_Media,N_PTDs,Ganho_LED,PFolga,PVE,D,Viabilidade_VE
0,Aveiro,Águeda,101,910.887701,244.320,0.268222,105715,0.477593,388,158.80800,50808.195148,5121.6,45845.403148,Viável
1,Aveiro,Albergaria-a-Velha,102,451.711801,28.020,0.062031,54540,0.469948,194,18.21300,26596.303834,2560.8,24053.716834,Viável
2,Aveiro,Anadia,103,657.071801,73.545,0.111928,55628,0.543009,223,47.80425,23387.762452,2943.6,20491.966702,Viável
3,Aveiro,Arouca,104,585.974400,115.720,0.197483,41884,0.527387,236,75.21800,18211.314133,3115.2,15171.332133,Viável
4,Aveiro,Aveiro,105,1055.192001,144.420,0.136866,197485,0.475475,509,93.87300,95298.999935,6718.8,88674.072935,Viável


In [14]:
# =========================================
# DROPDOWNS: Distrito -> Concelho
# =========================================

# Lista de distritos (ordenada)
lista_distritos = sorted(df_dashboard["Distrito"].dropna().unique())

# Dropdown de distrito
dropdown_distrito = widgets.Dropdown(
    options=lista_distritos,
    description="",
    layout=widgets.Layout(width="250px")
)

# Dropdown de concelho (atualizado dinamicamente)
dropdown_concelho = widgets.Dropdown(
    description="",
    layout=widgets.Layout(width="250px")
)

# Função para atualizar a lista de concelhos quando muda o distrito
def atualizar_concelhos(change):
    distrito = change["new"]

    lista_concelhos = sorted(
        df_dashboard.loc[df_dashboard["Distrito"] == distrito, "Concelho"]
        .dropna()
        .unique()
    )

    dropdown_concelho.options = lista_concelhos

    if lista_concelhos:
        dropdown_concelho.value = lista_concelhos[0]

# Ligar evento
dropdown_distrito.observe(atualizar_concelhos, names="value")

# Inicializar dropdowns
if lista_distritos:
    dropdown_distrito.value = lista_distritos[0]

In [15]:
def get_dados_concelho():
    # Obtém os dados do concelho selecionado nos dropdowns
    distrito = dropdown_distrito.value
    concelho = dropdown_concelho.value

    df_filtrado = df_dashboard.loc[
        (df_dashboard["Distrito"] == distrito) &
        (df_dashboard["Concelho"] == concelho)
        ]

    if df_filtrado.empty:
        return None

    return df_filtrado.iloc[0]

In [16]:
horas = list(range(24))

# Perfil horário normalizado (0–1) da iluminação pública ao longo do dia
# Valores elevados durante a noite e reduzidos durante o dia
perfil_iluminacao = np.array([
    0.75, 0.85, 0.95, 1.00, 1.00, 0.90, 0.60, 0.25,
    0.05, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.05,
    0.20, 0.50, 0.80, 0.95, 1.00, 1.00, 0.90, 0.80
])

assert len(perfil_iluminacao) == 24

In [17]:
def mostrar_metricas(row):
    estado = row["Viabilidade_VE"]
    emoji = "🟢" if estado == "Viável" else "🔴"
    cor = "green" if estado == "Viável" else "red"

    fator_potencia = 0.9

    capacidade_kw = row["Cap_PTD"] * fator_potencia
    folga_kw = row["PFolga"] * fator_potencia

    potencia_apos_led = row["P_IP_TOTAL"] - row["Ganho_LED"]

    # 🔥 saldo corrigido em kW
    saldo_kw = folga_kw + row["Ganho_LED"] - row["PVE"]

    display(Markdown(f"""
## {row['Concelho']} ({row['Distrito']})

### ⚡ Energia e Infraestrutura
- **Potência total:** {row['P_IP_TOTAL']:.2f} kW  
- **Potência ineficiente:** {row['P_IP_Inef']:.2f} kW  
- **Potência libertada (LED) pelas medidas de eficiência:** {row['Ganho_LED']:.2f} kW  
- **Potência após LED:** {potencia_apos_led:.2f} kW  
- **Capacidade PTD:** {capacidade_kw:.2f} kW  
- **Utilização média:** {row['Util_Media']:.2%}  
- **Folga da rede:** {folga_kw:.2f} kW  

### 🚗 Mobilidade Elétrica
- **Carga VE estimada:** {row['PVE']:.2f} kW  

---

## 🧮 Resultado Final (cenário com LED)

### **Saldo (D): {saldo_kw:.2f} kW**

### <span style="color:{cor}; font-size:18px;"><b>{emoji} {estado}</b></span>
"""))

In [18]:
def plot_perfil_horario(row):
    # Estimar consumo antes e depois da modernização LED
    consumo_antes = row["P_IP_TOTAL"] * perfil_iluminacao

    potencia_depois = max(row["P_IP_TOTAL"] - row["Ganho_LED"], 0)
    consumo_depois = potencia_depois * perfil_iluminacao

    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=horas,
        y=consumo_antes,
        mode="lines+markers",
        name="Antes (tecnologia convencional)"
    ))

    fig.add_trace(go.Scatter(
        x=horas,
        y=consumo_depois,
        mode="lines+markers",
        name="Depois (tecnologia LED)"
    ))

    fig.update_layout(
        title=f"Perfil horário de consumo — {row['Concelho']}",
        xaxis_title="Hora do dia",
        yaxis_title="Potência estimada (kW)",
        height=450
    )

    fig.show()

In [19]:
def plot_capacidade_viabilidade(row):
    fator_potencia = 0.9

    # Conversões para kW
    folga_kw = row["PFolga"] * fator_potencia
    ganho_led = row["Ganho_LED"]
    carga_ve = row["PVE"]

    # 🔥 saldo corrigido (não usar row["D"])
    saldo = folga_kw + ganho_led - carga_ve

    categorias = [
        "Folga atual",
        "Ganho LED",
        "Carga VE",
        "Saldo final"
    ]

    valores = [
        folga_kw,
        ganho_led,
        carga_ve,
        saldo
    ]

    cores = [
        "#B0BEC5",   # folga
        "#66BB6A",   # ganho LED
        "#EF5350",   # carga VE
        "#2E8B57" if saldo >= 0 else "#C0392B"  # saldo
    ]

    fig = go.Figure(
        data=[
            go.Bar(
                x=categorias,
                y=valores,
                text=[f"{v:.2f}" for v in valores],
                textposition="outside",
                marker_color=cores
            )
        ]
    )

    fig.update_layout(
        title=f"Viabilidade da rede após modernização LED — {row['Concelho']}",
        yaxis_title="Potência (kW)",
        template="plotly_white",
        height=460,
        showlegend=False,
        plot_bgcolor="white",
        paper_bgcolor="white",
        yaxis=dict(showgrid=True, gridcolor="#EAEAEA"),
        xaxis=dict(showgrid=False)
    )

    fig.show()

In [20]:
def plot_mapa_ptd(row):
    codigo = row["CodDistritoConcelho"]

    df_ptd = ptd_data[ptd_data["CodDistritoConcelho"] == codigo].copy()

    if df_ptd.empty:
        print("Sem PTDs para este concelho.")
        return

    coords = df_ptd["Coordenadas Geográficas"].astype(str).str.replace(" ", "").str.split(",", expand=True)
    df_ptd["lat"] = pd.to_numeric(coords[0], errors="coerce")
    df_ptd["lon"] = pd.to_numeric(coords[1], errors="coerce")
    df_ptd = df_ptd.dropna(subset=["lat", "lon"])

    if df_ptd.empty:
        print("Sem coordenadas válidas para este concelho.")
        return

    fig = px.scatter_map(
        df_ptd,
        lat="lat",
        lon="lon",
        hover_name="Tipo Construtivo" if "Tipo Construtivo" in df_ptd.columns else None,
        hover_data={
            "Código de Instalação": "Código de Instalação" in df_ptd.columns,
            "Potência instalada [kVA]": "Potência instalada [kVA]" in df_ptd.columns,
            "Nível de Utilização [%]": "Nível de Utilização [%]" in df_ptd.columns,
            "Concelho": "Concelho" in df_ptd.columns,
            "lat": False,
            "lon": False
        },
        zoom=11,
        height=500,
        title=f"Mapa dos PTDs — {row['Concelho']}",
        color_discrete_sequence=["#2E8B57"]
    )

    fig.update_layout(
        map_style="open-street-map",
        margin=dict(l=0, r=0, t=50, b=0)
    )

    fig.show()

In [21]:
output_dashboard = widgets.Output()

In [22]:
def atualizar_dashboard(*args):
    with output_dashboard:
        output_dashboard.clear_output()

        row = get_dados_concelho()
        if row is None:
            print("Nenhum concelho selecionado.")
            return

        # KPIs
        display(Markdown(
            "#### Este painel resume os principais indicadores energéticos e de capacidade da rede para o concelho selecionado."
        ))
        mostrar_metricas(row)
        display(Markdown("---"))

        # Viabilidade
        display(Markdown(
            "#### Este gráfico apresenta a folga atual da rede, o ganho estimado com a modernização LED, a carga dos veículos elétricos e o saldo final de viabilidade."
        ))
        plot_capacidade_viabilidade(row)
        display(Markdown("---"))

        # Perfil horário
        display(Markdown(
            "#### O perfil horário mostra a variação estimada do consumo de iluminação pública ao longo do dia, antes e depois da modernização para tecnologia LED."
        ))
        plot_perfil_horario(row)
        display(Markdown("---"))

        # Mapa
        display(Markdown(
            "#### O mapa seguinte mostra a localização dos PTDs do concelho selecionado, identificando a infraestrutura onde poderão ser analisados potenciais pontos de carregamento."
        ))
        plot_mapa_ptd(row)

In [23]:
dropdown_distrito.observe(atualizar_dashboard, names="value")
dropdown_concelho.observe(atualizar_dashboard, names="value")

display(widgets.HBox([dropdown_distrito, dropdown_concelho]))
display(output_dashboard)

atualizar_dashboard()

Output()